# Can we beat 1/N? Robust long-only portfolios (out-of-sample)

Same train/test harness (3 months train, 1 month held out), but instead of sweeping $\gamma$ on noisy mean-variance, we test **robust portfolios that lean on $\Sigma$ and ignore or down-weight $\mu$** — the thing that was being overfit. Hypothesis: $\Sigma$-based construction beats both mean-variance *and* equal-weight on a risk-adjusted basis.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from stonks import get_prices, to_returns

%matplotlib inline


## Parameters


In [ ]:
TOP_N = 500
TRAIN_MONTHS = 3
TEST_MONTHS = 1
PERIOD = "2y"
GAMMA = 20.0        # for the mean-variance strategies
MU_SHRINK = 0.3     # shrink factor on mu
LW_ALPHA = 0.5      # Ledoit-Wolf shrinkage on Sigma
torch.manual_seed(0)


## Fetch, split train/test


In [ ]:
prices = get_prices(top_n=TOP_N, period=PERIOD, interval="1d", field="close")
last = pd.Timestamp(prices.columns[-1])
returns = to_returns(prices.loc[:, prices.columns >= last - pd.DateOffset(months=TRAIN_MONTHS + TEST_MONTHS)]).dropna()
test_start = last - pd.DateOffset(months=TEST_MONTHS)
train = returns.loc[:, returns.columns < test_start]
test  = returns.loc[:, returns.columns >= test_start]
N = returns.shape[0]
tr = train.to_numpy(dtype=float); te = test.to_numpy(dtype=float)
print(f"N={N}  train {train.shape}  test {test.shape}")


## $\mu$, $\Sigma$ on the training window


In [ ]:
mu = tr.mean(axis=1)
Xc = tr - mu[:, None]
Sigma = (Xc @ Xc.T) / (tr.shape[1] - 1)
sigma = np.sqrt(np.diag(Sigma))
# Ledoit-Wolf-style target: shrink Sigma toward the average-variance diagonal.
F = np.mean(np.diag(Sigma)) * np.eye(N)
Sigma_LW = (1 - LW_ALPHA) * Sigma + LW_ALPHA * F
mu_t = torch.tensor(mu, dtype=torch.float64)
Sig_t = torch.tensor(Sigma, dtype=torch.float64)


## Strategies

- **1/N** — equal weight; no estimation error (the benchmark).
- **min-variance** — ignore $\mu$, minimize $w^\top\Sigma w$.
- **risk parity (1/σ)** — $w_i \propto 1/\sigma_i$; equal risk contribution, uses only vols.
- **mean-variance $\gamma=20$** — uses $\mu$ directly (the overfit baseline from before).
- **shrink-$\mu$** — $\mu_s = 0.3\,\hat\mu$ then mean-variance; mild tilt.
- **LW min-variance** — min-variance on the shrunk $\Sigma_{\text{LW}}$.


In [ ]:
def optimize(mu_vec, Sig_mat, sense="mv"):
    """Long-only softmax optimizer. sense='mv' -> mean-variance (GAMMA); 'minvar' -> min variance."""
    Sm = torch.tensor(Sig_mat, dtype=torch.float64)
    mv = torch.tensor(mu_vec, dtype=torch.float64)
    z = torch.zeros(N, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.Adam([z], lr=0.5)
    for _ in range(8000):
        opt.zero_grad()
        w = torch.softmax(z, dim=0)
        loss = (w @ Sm @ w) if sense == "minvar" else -(mv @ w - 0.5 * GAMMA * w @ Sm @ w)
        loss.backward(); opt.step()
    return torch.softmax(z, dim=0).detach().numpy()

strategies = {
    "1/N":                  np.ones(N) / N,
    "min-variance":         optimize(np.zeros(N), Sigma, "minvar"),
    "risk parity (1/sigma)": (1 / sigma) / (1 / sigma).sum(),
    f"mean-var g={GAMMA:g}":  optimize(mu, Sigma, "mv"),
    f"shrink-mu c={MU_SHRINK}": optimize(MU_SHRINK * mu, Sigma, "mv"),
    "LW min-variance":      optimize(np.zeros(N), Sigma_LW, "minvar"),
}
print("built", len(strategies), "portfolios on the train window")


## Out-of-sample results (held-out month)


In [ ]:
def perf(w):
    rp = w @ te
    profit = np.prod(1 + rp) - 1
    vol = rp.std() * np.sqrt(252)
    sharpe = rp.mean() / rp.std() * np.sqrt(252)
    return profit, vol, sharpe

base = perf(strategies["1/N"])[2]   # 1/N Sharpe, for tagging
rows = []
for name, w in strategies.items():
    p, v, s = perf(w)
    rows.append({"strategy": name, "holdings": int((w > 1e-3).sum()),
                 "test_profit%": p * 100, "test_vol%": v * 100, "test_Sharpe": s,
                 "beats_1N_Sharpe": s > base})
results = pd.DataFrame(rows).sort_values("test_Sharpe", ascending=False).reset_index(drop=True)
results


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, w in strategies.items():
    cum = np.cumprod(1 + w @ te) - 1
    ax.plot(test.columns, cum, lw=2.5 if name == "1/N" else 1.2, label=name)
ax.axhline(0, color="gray", lw=0.5)
ax.set_ylabel("cumulative return"); ax.set_title("Held-out month: cumulative return by strategy")
ax.legend()


### Takeaway

- **The diagnosis holds.** Anything that used $\mu$ (mean-variance, even shrunk-$\mu$) lost money OOS; the robust portfolios that **ignore $\mu$ and use $\Sigma$** survived and shone.
- **1/N made the most raw profit** this month — but it was a strong up-month where full diversification (full market beta) captured all the upside.
- **On a risk-adjusted basis, $\Sigma$-based wins**: risk parity and LW min-variance took far less risk (~7-9% vol vs 14%) for similar return, so their **Sharpe beat 1/N**. In a flat or down month — where 1/N's full beta hurts — they'd likely beat it on profit too.
- **Metric matters.** Rank by profit and 1/N wins (up-month); rank by Sharpe and risk parity wins. For a long-only investor who can't short, risk parity / min-variance give you most of the return with much less risk — the honest improvement over both mean-variance *and* 1/N.
- Still a single test month; walk-forward over many regimes is the real test.
